In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from PIL import Image
import os
import numpy as np
import plotly.graph_objects as go
from scipy.spatial import cKDTree
from sklearn.decomposition import PCA
import torch.nn as nn

def detect_background_color(img, samples_per_edge=100):
    """
    Safely detect the background color by analyzing image edges.
    Uses a more robust sampling approach that avoids zero-step slicing.
    
    Args:
        img: Input image in BGR format
        samples_per_edge: Number of samples to take from each edge
        
    Returns:
        tuple: RGB color values of detected background
    """
    height, width = img.shape[:2]
    
    # Ensure we take at least one sample per edge
    samples_per_edge = max(1, min(samples_per_edge, min(height, width)))
    
    # Calculate step sizes (ensure non-zero)
    width_step = max(1, width // samples_per_edge)
    height_step = max(1, height // samples_per_edge)
    
    # Sample pixels from edges
    edge_pixels = []
    
    # Sample top and bottom edges
    for x in range(0, width, width_step):
        edge_pixels.append(img[0, x])        # Top edge
        edge_pixels.append(img[-1, x])       # Bottom edge
    
    # Sample left and right edges
    for y in range(0, height, height_step):
        edge_pixels.append(img[y, 0])        # Left edge
        edge_pixels.append(img[y, -1])       # Right edge
    
    # Convert to numpy array
    edge_pixels = np.array(edge_pixels)
    
    # Use K-means clustering to find dominant color
    kmeans = KMeans(n_clusters=3, n_init=10)
    kmeans.fit(edge_pixels)
    
    # Get the most frequent color cluster
    unique, counts = np.unique(kmeans.labels_, return_counts=True)
    dominant_cluster = unique[np.argmax(counts)]
    background_color = kmeans.cluster_centers_[dominant_cluster].astype(int)
    
    # Convert from BGR to RGB
    return tuple(background_color[::-1])

def detect_shadows_enhanced(img, background_color):
    """
    Enhanced shadow detection that preserves subtle lighting transitions and reflections.
    This function analyzes both global and local lighting patterns to identify shadows
    while being careful not to misclassify object details as shadows.
    
    Args:
        img: Input image in BGR format
        background_color: RGB tuple of the background color
    Returns:
        numpy.ndarray: Shadow mask where 1 indicates shadow pixels
    """
    # First, convert background color from RGB to BGR for OpenCV
    bg_color = background_color[::-1]
    
    # Create a background image matching our detected background color
    background = np.full_like(img, bg_color)
    
    # Convert both images to LAB color space for better lighting analysis
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    bg_lab = cv2.cvtColor(background, cv2.COLOR_BGR2LAB)
    
    # Extract L (lightness) channels
    l_img = lab[:,:,0].astype(np.float32)
    l_bg = bg_lab[:,:,0].astype(np.float32)
    
    # Calculate local average of lightness using multiple scales
    shadow_masks = []
    for kernel_size in [(21, 21), (41, 41)]:  # Multiple scales for different shadow sizes
        # Calculate local lightness average
        local_mean = cv2.GaussianBlur(l_img, kernel_size, 0)
        
        # Calculate lightness difference from background
        l_diff = np.abs(l_img - l_bg)
        local_diff = np.abs(local_mean - l_bg)
        
        # Create adaptive threshold based on local contrast
        threshold = np.mean(l_diff) * 0.5 + local_diff * 0.2
        
        # Identify shadow regions
        shadow = (l_diff < threshold) & (l_img < l_bg)
        shadow_masks.append(shadow)
    
    # Combine shadow masks from different scales
    shadow_mask = np.logical_or.reduce(shadow_masks)
    
    # Analyze color differences to prevent misclassifying colored regions as shadows
    a_diff = np.abs(lab[:,:,1] - bg_lab[:,:,1])
    b_diff = np.abs(lab[:,:,2] - bg_lab[:,:,2])
    color_diff = np.sqrt(a_diff**2 + b_diff**2)
    
    # Only keep shadow pixels where color difference is small
    shadow_mask &= (color_diff < 30)
    
    # Clean up the mask
    kernel = np.ones((3,3), np.uint8)
    shadow_mask = cv2.morphologyEx(shadow_mask.astype(np.uint8), 
                                 cv2.MORPH_CLOSE, kernel)
    shadow_mask = cv2.morphologyEx(shadow_mask.astype(np.uint8), 
                                 cv2.MORPH_OPEN, kernel)
    
    return shadow_mask

def refined_background_removal(image_input, background_color=None, edge_smoothing=5, preserve_whites=True):
    """
    Advanced background removal with automatic background color detection and
    improved edge handling.
    
    Args:
        image_input: Path to input image or Image.Image or numpy array
        background_color: RGB tuple or None for auto-detection
        edge_smoothing: Amount of edge smoothing (higher = smoother)
        preserve_whites: Whether to preserve white colors in the object
    """
    def create_color_range_mask(img, target_color, tolerance=20):  # Reduced tolerance
        """Create a more precise mask for colors within tolerance of target color"""
        target_bgr = target_color[::-1]

        # Convert to LAB color space for better color similarity matching
        lab_image = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        lab_target = cv2.cvtColor(np.uint8([[target_bgr]]), cv2.COLOR_BGR2LAB)[0,0]

        # Create bounds with tolerance in LAB space
        lower_bound = np.array([max(0, c - tolerance) for c in lab_target])
        upper_bound = np.array([min(255, c + tolerance) for c in lab_target])

        # Create mask in LAB space
        mask = cv2.inRange(lab_image, lower_bound, upper_bound)

        # Clean up the mask
        kernel = np.ones((3,3), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        return mask

    def smooth_edges(mask, smooth_factor):
        """Apply edge smoothing with outline removal"""
        # Convert to float
        mask_float = mask.astype(np.float32) / 255.0
        
        # Apply erosion first to remove thin outlines
        kernel = np.ones((2,2), np.uint8)
        eroded = cv2.erode(mask_float, kernel, iterations=1)
        
        # Multi-scale smoothing
        smoothed = np.zeros_like(mask_float)
        weights_sum = 0
        
        for i in range(1, 4):
            kernel_size = smooth_factor * 2 * i + 1
            current_smooth = cv2.GaussianBlur(eroded, 
                                            (kernel_size, kernel_size), 
                                            0)
            weight = 1.0 / i
            smoothed += current_smooth * weight
            weights_sum += weight
        
        smoothed /= weights_sum
        
        # Threshold the result to make edges cleaner
        smoothed = np.where(smoothed > 0.5, 1.0, 0.0)
        
        return (smoothed * 255).astype(np.uint8)

    def preserve_white_details(img, mask, threshold=250):
        """
        Preserve white details with proper handling of edge cases and division.
        
        Args:
            img: Input image in BGR format
            mask: Binary mask
            threshold: Brightness threshold for white detection
        """
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Create adaptive threshold with safety checks
        local_mean = cv2.GaussianBlur(gray, (15, 15), 0)
        
        # Avoid division by zero and invalid values
        local_mean = np.clip(local_mean, 1, 254)  # Ensure no zeros or 255s
        adjustment = np.clip((255 - local_mean) * 0.05, 0, threshold)
        local_threshold = threshold - adjustment
        
        # Create white mask with safety checks
        white_areas = gray > local_threshold
        
        # Reduce connectivity to main object
        kernel = np.ones((2,2), np.uint8)
        dilated_mask = cv2.dilate(mask, kernel, iterations=1)
        preserved_whites = white_areas & dilated_mask
        
        # Clean up artifacts
        preserved_whites = cv2.morphologyEx(preserved_whites.astype(np.uint8), 
                                        cv2.MORPH_OPEN, kernel)
        preserved_whites = cv2.morphologyEx(preserved_whites, 
                                        cv2.MORPH_CLOSE, kernel)
        
        return mask | preserved_whites

    def shrink_mask(mask, shrink_percent=1):
        """
        Shrink the mask by a percentage of its dimensions
        Args:
            mask: Binary mask
            shrink_percent: Percentage to shrink (1 = 1%)
        Returns:
            Shrunk mask
        """
        # Get mask dimensions
        height, width = mask.shape[:2]
        
        # Calculate pixels to shrink on each side
        shrink_pixels_y = int(height * (shrink_percent / 100))
        shrink_pixels_x = int(width * (shrink_percent / 100))
        
        # Ensure at least 1 pixel if percentage is too small
        shrink_pixels_y = max(1, shrink_pixels_y)
        shrink_pixels_x = max(1, shrink_pixels_x)
        
        # Create structuring element for erosion
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * shrink_pixels_x + 1, 2 * shrink_pixels_y + 1)
        )
        
        # Erode the mask
        shrunk_mask = cv2.erode(mask, kernel, iterations=1)
        
        return shrunk_mask

    # Main processing pipeline
    try:
        print("Loading image...")
        # Input validation and conversion
        if isinstance(image_input, str):
            print("It's a file path - load the image")
            if not os.path.exists(image_input):
                raise ValueError(f"Image file not found: {image_input}")
            image = cv2.imread(image_input)
            if image is None:
                raise ValueError(f"Failed to load image from {image_input}")
        elif isinstance(image_input, np.ndarray):
            print("It's already a numpy array - verify format")
            if len(image_input.shape) != 3 or image_input.shape[2] != 3:
                raise ValueError("Image array must be a 3-channel color image")
            image = image_input
        elif isinstance(image_input, Image.Image):
            
            print("Convert PIL Image to numpy array in BGR format")
            image_array = np.array(image_input)
            image = cv2.cvtColor(image_array, cv2.COLOR_RGB2BGR)
        else:
            raise TypeError("Image input must be either a file path, numpy array, or PIL Image")
        
        print(image.__class__)
        # Handle background color detection
        if background_color is None:
            print("Detecting background color...")
            detected_color = detect_background_color(image)
            print(f"Detected background color (RGB): {detected_color}")
            bg_color = detected_color
        else:
            bg_color = background_color
        
        print("Detecting shadows...")
        shadow_mask = detect_shadows_enhanced(image, bg_color)
        
        print("Creating color-based mask...")
        color_mask = create_color_range_mask(image, bg_color)
        
        # Combine color and shadow masks
        combined_mask = (color_mask | shadow_mask)

        
        
        # Invert mask (we want to keep the object, not the background)
        object_mask = cv2.bitwise_not(combined_mask)
        
        # Preserve white details if requested
        if preserve_whites:
            print("Preserving white details...")
            object_mask = preserve_white_details(image, object_mask)

        
        # Apply edge smoothing
        if edge_smoothing > 0:
            print("Smoothing edges...")
            object_mask = smooth_edges(object_mask, edge_smoothing)

        # Shrinking mask
        print("Shrinking mask...")
        object_mask = shrink_mask(object_mask, shrink_percent=0.5)
        
        
        # Create output images
        alpha = object_mask
        rgba = cv2.cvtColor(image, cv2.COLOR_BGR2BGRA)
        rgba[:, :, 3] = alpha
        
        result = image.copy()
        result[alpha == 0] = [0, 0, 0]
        
        # Convert to RGB for display
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
        
        checkered = np.zeros((image.shape[0], image.shape[1], 3), dtype=np.uint8)
        checkered[::20, ::20] = [200, 200, 200]
        checkered[10::20, 10::20] = [200, 200, 200]
        
        alpha_3d = alpha[:,:,np.newaxis] / 255.0
        blended = (image_rgb * alpha_3d + checkered * (1 - alpha_3d)).astype(np.uint8)
        
        return image_rgb, alpha, result_rgb, blended, rgba
        
    except Exception as e:
        print(f"Error during processing: {str(e)}")
        return None

In [2]:
front_results = refined_background_removal(
            "../static/uploads/Chair/front.jpg",
            background_color=None,  # Enable auto-detection
            edge_smoothing=5,
            preserve_whites=True
        )

front_feature = front_results[2];

back_results = refined_background_removal(
            "../static/uploads/Chair/back.jpg",
            background_color=None,  # Enable auto-detection
            edge_smoothing=5,
            preserve_whites=True
        )

back_feature = back_results[2];
top_results = refined_background_removal(
            "../static/uploads/Chair/top.jpg",
            background_color=None,  # Enable auto-detection
            edge_smoothing=5,
            preserve_whites=True
        )

top_feature = top_results[2];
top_binary_mask = top_results[1];

Loading image...
It's a file path - load the image
<class 'numpy.ndarray'>
Detecting background color...


c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...
Loading image...
It's a file path - load the image
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...


c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Loading image...
It's a file path - load the image
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...


In [3]:
import torch
import os
import cv2
import numpy as np
from PIL import Image
import plotly.graph_objects as go
from transformers import pipeline
 
# Get depth map using Depth Anything model
device = "cuda" if torch.cuda.is_available() else "cpu"
depth_model = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-base-hf", device=device)

In [4]:
def get_depth_map(result_rgb):
    # Use result_rgb directly for depth estimation (it's already in RGB format)
    image_pil = Image.fromarray(result_rgb)
    depth_predictions = depth_model(image_pil)
    depth_map = np.array(depth_predictions["depth"])

    # Process depth map with dimensions from result_rgb
    height, width = result_rgb.shape[:2]
    depth_map_resized = cv2.resize(depth_map, (width, height))

    return depth_map_resized

front_depth_map = get_depth_map(front_feature)
back_depth_map = get_depth_map(back_feature)
top_depth_map = get_depth_map(top_feature)

In [5]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KDTree
import plotly.graph_objects as go
import numpy as np
import sklearn
import scipy

In [21]:
def load_and_normalize_images(image_paths, target_size=256):
    """
    Load and normalize all images to the specified target size while maintaining aspect ratios.
    Uses background color detection for padding.

    The function:
    1. Loads each image
    2. Detects the background color
    3. Resizes while maintaining aspect ratio
    4. Creates a square canvas with the detected background color
    5. Centers the resized image in the canvas
    """
    print("\nLoading and normalizing images...")

    images = {}
    for view_type, path in image_paths.items():
        if path is None:
            continue

        # Load the image
        img = cv2.imread(path)
        if img is None:
            raise ValueError(f"Failed to load image: {path}")

        # Detect background color (returns RGB)
        bg_color_rgb = detect_background_color(img)
        # Convert RGB to BGR for OpenCV
        bg_color_bgr = bg_color_rgb[::-1]
        print(f"{view_type.capitalize()} view background color (RGB): {bg_color_rgb}")

        # Calculate aspect ratio preserving dimensions
        h, w = img.shape[:2]
        aspect = w / h

        if aspect > 1:
            new_w = target_size
            new_h = int(target_size / aspect)
        else:
            new_h = target_size
            new_w = int(target_size * aspect)

        # Resize image
        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

        # Create square canvas with detected background color (in BGR)
        square_img = np.full(
            (target_size, target_size, 3), bg_color_bgr, dtype=np.uint8
        )

        # Calculate padding to center the image
        pad_y = (target_size - new_h) // 2
        pad_x = (target_size - new_w) // 2

        # Place resized image in center
        square_img[pad_y : pad_y + new_h, pad_x : pad_x + new_w] = resized

        images[view_type] = square_img
        print(
            f"{view_type.capitalize()} view normalized size: {square_img.shape[1]}x{square_img.shape[0]}"
        )

    return images, target_size


def remove_outliers(
    fig, spatial_strictness=1.0, color_threshold=0.05, min_cluster_size=5
):
    """
    Removes outliers from a 3D point cloud based on both spatial distribution and color.

    Parameters:
    -----------
    fig : plotly.graph_objects.Figure
        Input figure containing the 3D scatter plot
    spatial_strictness : float, default=1.0
        Controls how strict the spatial clustering should be:
        - Lower values (e.g., 0.5) are more lenient and keep more points
        - Higher values (e.g., 2.0) are stricter and remove more points
        - 1.0 is the default balanced setting
    color_threshold : float, default=0.05
        The minimum percentage (0.0 to 1.0) of points a color cluster needs to be considered valid
    min_cluster_size : int, default=5
        Minimum number of points needed to form a valid cluster

    Returns:
    --------
    plotly.graph_objects.Figure
        New figure with outliers removed
    """

    def calculate_dynamic_dbscan_params(points, strictness):
        # Create KD-tree for efficient nearest neighbor search
        point_tree = KDTree(points)
        # Get distances to k nearest neighbors
        distances, _ = point_tree.query(points, k=6)  # k=6 gives 5 neighbors + self

        # Calculate average distance to nearest neighbors
        avg_distance = np.mean(distances[:, 1:])  # Exclude distance to self

        # Adjust eps based on strictness parameter
        # Lower strictness = larger eps = more lenient clustering
        eps = avg_distance * (2.0 / strictness)

        # Calculate dynamic minimum samples
        # Base number is min_cluster_size, scaled by log of point count
        point_count_factor = (
            np.log10(len(points)) / 4
        )  # Divide by 4 to make it less strict
        min_samples = max(min_cluster_size, int(point_count_factor * min_cluster_size))

        return eps, min_samples

    def analyze_color_clusters(colors, threshold):
        """
        Analyzes color distribution using color distance and density-based grouping.

        Parameters:
        -----------
        colors : numpy.ndarray
            Array of RGB colors
        threshold : float
            Threshold for considering a color group significant (0.0 to 1.0)

        Returns:
        --------
        numpy.ndarray
            Boolean mask of valid colors
        """

        def color_distance(c1, c2):
            """
            Calculates a weighted color distance that's more perceptually accurate.
            Gives more weight to differences in the same color channel.
            """
            # Convert to float to avoid overflow
            c1 = c1.astype(float)
            c2 = c2.astype(float)

            # Calculate channel differences
            dr = c1[0] - c2[0]
            dg = c1[1] - c2[1]
            db = c1[2] - c2[2]

            # Weight the differences (giving more weight to same-channel differences)
            return np.sqrt(2 * dr * dr + 4 * dg * dg + 3 * db * db)

        # Find the dominant color (the color that appears most frequently)
        unique_colors, color_counts = np.unique(colors, axis=0, return_counts=True)
        dominant_color = unique_colors[np.argmax(color_counts)]

        # Calculate distances from each point to the dominant color
        distances = np.array(
            [color_distance(color, dominant_color) for color in colors]
        )

        # Calculate adaptive threshold based on the distribution of distances
        distance_threshold = np.percentile(distances, threshold * 100)

        # Create mask for colors within threshold
        valid_colors = distances <= distance_threshold

        return valid_colors

    # Extract points and colors from figure
    trace = fig.data[0]
    points = np.column_stack([trace.x, trace.y, trace.z])
    colors = np.array(
        [list(map(int, c.strip("rgb()").split(","))) for c in trace.marker.color]
    )

    # Scale the points
    scaler = StandardScaler()
    xy_scaled = scaler.fit_transform(points[:, :2])
    z_scaled = scaler.fit_transform(points[:, 2:3]) * 2  # Weight depth more
    points_scaled = np.column_stack([xy_scaled, z_scaled])

    # Get dynamic parameters and perform spatial clustering
    eps, min_samples = calculate_dynamic_dbscan_params(
        points_scaled, spatial_strictness
    )
    spatial_clusters = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1).fit_predict(
        points_scaled
    )

    # Get valid points from spatial clustering
    valid_spatial = spatial_clusters != -1

    # Get valid points from color analysis
    valid_colors = analyze_color_clusters(colors, color_threshold)

    # Combine both masks
    valid_points = valid_spatial & valid_colors

    # Print final statistics
    total_points = len(points)
    kept_points = np.sum(valid_points)

    print(f"Outliers removed: {total_points - kept_points}")

    # Create new figure with cleaned points
    cleaned_fig = go.Figure(
        data=[
            go.Scatter3d(
                x=points[valid_points, 0],
                y=points[valid_points, 1],
                z=points[valid_points, 2],
                mode="markers",
                marker=dict(
                    size=trace.marker.size,
                    color=[f"rgb({r},{g},{b})" for r, g, b in colors[valid_points]],
                    opacity=trace.marker.opacity,
                ),
            )
        ]
    )

    # Copy layout from original figure
    cleaned_fig.update_layout(fig.layout)

    return cleaned_fig


# def fill_gaps_with_interpolation(front_points, back_points, num_interpolation_steps=5):
#     """
#     Fills gaps between front and back point clouds using geometric interpolation.

#     This function:
#     1. Identifies corresponding regions between views
#     2. Creates interpolated points to fill gaps
#     3. Maintains smooth transitions in both geometry and color
#     """
#     def find_corresponding_points(source_points, target_points, max_distance):
#         """Finds pairs of points that likely correspond between views."""
#         tree = sklearn.neighbors.KDTree(target_points)
#         distances, indices = tree.query(source_points, k=1)
#         valid_pairs = distances < max_distance
#         return source_points[valid_pairs], target_points[indices[valid_pairs]]

#     def create_interpolated_points(p1, p2, steps):
#         """Creates smooth interpolation between two points."""
#         # Use cubic interpolation for smoother transitions
#         t = np.linspace(0, 1, steps)
#         # Hermite interpolation for position
#         points = []
#         for ti in t:
#             # Cubic interpolation weight
#             h1 = 2*ti**3 - 3*ti**2 + 1
#             h2 = -2*ti**3 + 3*ti**2
#             # Create interpolated point
#             point = h1*p1 + h2*p2
#             points.append(point)
#         return np.array(points)

#     # Parameters for finding corresponding points
#     max_correspondence_distance = np.mean([
#         np.std(front_points, axis=0),
#         np.std(back_points, axis=0)
#     ]) * 0.5

#     # Find corresponding points between views
#     front_corresp, back_corresp = find_corresponding_points(
#         front_points, back_points, max_correspondence_distance)

#     # Generate interpolated points between corresponding pairs
#     interpolated_points = []
#     for fp, bp in zip(front_corresp, back_corresp):
#         new_points = create_interpolated_points(fp, bp, num_interpolation_steps)
#         interpolated_points.extend(new_points)


#     return np.array(interpolated_points)
def create_object_mesh(image_input, depth_threshold=0.25, target_size=256):
    """
    Creates a 3D mesh from either an image path or a numpy array using refined background removal.
    """
    # Input validation and conversion
    if isinstance(image_input, str):
        # Normalize the image first using our original normalization function
        images, _ = load_and_normalize_images({"input": image_input}, target_size)
        image = images["input"]
    elif isinstance(image_input, np.ndarray):
        image = image_input
    elif isinstance(image_input, Image.Image):
        image_array = np.array(image_input)
        image = cv2.cvtColor(image_array, cv2.COLOR_RGB2BGR)
    else:
        raise TypeError(
            "Image input must be either a file path, numpy array, or PIL Image"
        )

    try:

        # After background removal
        _, alpha_mask, result_rgb, _, _ = refined_background_removal(image)
        print(f"\nAfter background removal:")
        print(f"Result RGB shape: {result_rgb.shape}")
        print(f"Alpha mask shape: {alpha_mask.shape}")
        print(f"Non-zero alpha pixels: {np.count_nonzero(alpha_mask)}")

        # Get depth map
        device = "cuda" if torch.cuda.is_available() else "cpu"
        depth_model = pipeline(
            "depth-estimation",
            model="depth-anything/Depth-Anything-V2-base-hf",
            device=device,
        )
        depth_predictions = depth_model(Image.fromarray(result_rgb))
        depth_map = np.array(depth_predictions["depth"])

        print(f"\nDepth map statistics:")
        print(f"Depth map shape: {depth_map.shape}")
        print(f"Depth range: {depth_map.min():.3f} to {depth_map.max():.3f}")

        # Process depth map
        height, width = result_rgb.shape[:2]
        depth_map_resized = cv2.resize(depth_map, (width, height))
        alpha_mask_resized = cv2.resize(
            alpha_mask, (depth_map_resized.shape[1], depth_map_resized.shape[0])
        )
        depth_mask = alpha_mask_resized > 0

        print(f"\nAfter resizing:")
        print(f"Resized depth map shape: {depth_map_resized.shape}")
        print(f"Valid mask points: {np.count_nonzero(depth_mask)}")

        # Normalize depth map
        depth_norm = (depth_map_resized - depth_map_resized.min()) / (
            depth_map_resized.max() - depth_map_resized.min()
        )
        print(
            f"\nNormalized depth range: {depth_norm.min():.3f} to {depth_norm.max():.3f}"
        )

        # Create points
        x_coords, y_coords = np.meshgrid(np.arange(width), np.arange(height))
        valid_points = depth_mask & (depth_norm > depth_threshold)

        x_filtered = x_coords[valid_points].tolist()
        y_filtered = y_coords[valid_points].tolist()
        z_filtered = depth_norm[valid_points].tolist()
        colors_filtered = result_rgb[valid_points].tolist()

        print(f"\nPoint generation results:")
        print(f"Total possible points: {width * height}")
        print(f"Points after masking: {np.count_nonzero(depth_mask)}")
        print(f"Final points after depth threshold: {len(x_filtered)}")

        # Create the figure and return
        fig = go.Figure(
            data=[
                go.Scatter3d(
                    x=x_filtered,
                    y=y_filtered,
                    z=z_filtered,
                    mode="markers",
                    marker=dict(
                        size=2,
                        color=[f"rgb({r},{g},{b})" for r, g, b in colors_filtered],
                        opacity=1,
                    ),
                )
            ]
        )

        print("\n=== Mesh Creation Complete ===")
        return fig

    except Exception as e:
        print(f"Error during mesh creation: {str(e)}")
        return None


def combine_views_with_gap_filling(combined_fig, point_spacing=5):
    """
    Creates straight parallel lines with consistent, vibrant colors.
    """
    def create_fade_color(original_color, fade_factor):
        """
        Creates a color transition that maintains vibrancy.
        Instead of fading to black, we keep colors bright but adjust their intensity.
        """
        original_color = np.array(original_color, dtype=float)
        
        # Ensure fade factor never goes below 0.3 to maintain color visibility
        fade_factor = max(0.3, fade_factor)
        
        # Apply gamma correction for natural color appearance
        gamma = 2.2
        color_linear = (original_color / 255.0) ** gamma
        
        # Calculate faded color while maintaining vibrancy
        faded_linear = color_linear * fade_factor
        faded_color = 255.0 * (faded_linear ** (1.0/gamma))
        
        return np.clip(faded_color, 0, 255).astype(np.uint8)

    def fill_straight(points, colors, is_front):
        """
        Creates straight lines from points toward center with vibrant colors.
        """
        filled_points = []
        filled_colors = []
        
        center_z = 0.5  # Center point
        
        for point, color in zip(points, colors):
            # Calculate distance to center
            z_distance = abs(center_z - point[2])
            
            # Calculate number of points needed
            num_points = max(2, int(z_distance / (point_spacing * 0.01)) + 1)
            
            # Create sequence of points
            for i in range(num_points):
                progress = i / (num_points - 1)
                
                # Create new point
                new_point = point.copy()
                
                # Only modify z coordinate
                new_point[2] = point[2] + progress * (center_z - point[2])
                
                # Calculate color with maintained vibrancy
                # Use a gentler fade that never goes too dark
                fade_factor = 1.0 - (progress * 0.5)  # Only fade to 50% intensity
                faded_color = create_fade_color(color, fade_factor)
                
                filled_points.append(new_point)
                filled_colors.append(faded_color)
        
        if filled_points:
            return np.array(filled_points), np.array(filled_colors)
        return np.array([]), np.array([])

    # Extract points and colors
    points = np.column_stack([
        combined_fig.data[0].x,
        combined_fig.data[0].y,
        combined_fig.data[0].z
    ])
    colors = np.array([list(map(int, c.strip('rgb()').split(','))) 
                      for c in combined_fig.data[0].marker.color])
    
    # Split front and back
    split_x = np.median(points[:, 0])
    front_mask = points[:, 0] > split_x
    back_mask = ~front_mask
    
    # Fill from both surfaces
    front_filled_points, front_filled_colors = fill_straight(
        points[front_mask], colors[front_mask], True)
    back_filled_points, back_filled_colors = fill_straight(
        points[back_mask], colors[back_mask], False)
    
    print(f"\nFill Results:")
    print(f"Front fill points: {len(front_filled_points)}")
    print(f"Back fill points: {len(back_filled_points)}")
    
    # Combine all points
    all_points = np.vstack([
        points,  # Original points
        front_filled_points,
        back_filled_points
    ])
    all_colors = np.vstack([
        colors,  # Original colors
        front_filled_colors,
        back_filled_colors
    ])
    
    # Create visualization
    filled_fig = go.Figure(data=[go.Scatter3d(
        x=all_points[:, 0],
        y=all_points[:, 1],
        z=all_points[:, 2],
        mode='markers',
        marker=dict(
            size=5,
            color=[f'rgb({r},{g},{b})' for r,g,b in all_colors],
            opacity=combined_fig.data[0].marker.opacity
        )
    )])
    
    filled_fig.update_layout(combined_fig.layout)
    
    return filled_fig

def combine_front_back_meshes(
    front_fig, back_fig, depth_threshold=0.25, sampling_rate=1.0
):
    """Combines front and back meshes with detailed debugging information."""
    print("\n=== Starting Mesh Combination ===")

    def extract_points(fig, view_name):
        if fig is None or not fig.data:
            print(f"No data found for {view_name} view")
            return [], [], [], []

        trace = fig.data[0]
        points = (
            np.array(trace.x),
            np.array(trace.y),
            np.array(trace.z),
            [c.strip("rgb()").split(",") for c in trace.marker.color],
        )
        print(f"\n{view_name} view initial points: {len(points[0])}")
        return points

    def sample_points(x, y, z, colors, rate, view_name):
        n_points = len(x)
        n_sample = int(n_points * rate)
        print(f"\nSampling {view_name} view:")
        print(f"Original points: {n_points}")
        print(f"Target points: {n_sample}")

        if n_sample >= n_points:
            return x, y, z, colors

        indices = np.random.choice(n_points, n_sample, replace=False)
        indices.sort()

        sampled = (x[indices], y[indices], z[indices], [colors[i] for i in indices])
        print(f"Points after sampling: {len(sampled[0])}")
        return sampled

    # Extract points
    front_x, front_y, front_z, front_colors = extract_points(front_fig, "Front")
    back_x, back_y, back_z, back_colors = extract_points(back_fig, "Back")

    if len(front_x) == 0 and len(back_x) == 0:
        print("No points found in either view")
        return None

    # Sample points if needed
    front_x, front_y, front_z, front_colors = sample_points(
        front_x, front_y, front_z, front_colors, sampling_rate, "Front"
    )
    back_x, back_y, back_z, back_colors = sample_points(
        back_x, back_y, back_z, back_colors, sampling_rate, "Back"
    )

    # Process dimensions
    original_width = max(
        np.max(front_x) if len(front_x) > 0 else 0,
        np.max(back_x) if len(back_x) > 0 else 0,
    )
    original_height = max(
        np.max(front_y) if len(front_y) > 0 else 0,
        np.max(back_y) if len(back_y) > 0 else 0,
    )

    print(f"\nDimensions:")
    print(f"Original width: {original_width}")
    print(f"Original height: {original_height}")
    print(
        f"Aspect ratio: {original_width/original_height if original_height != 0 else 'N/A'}"
    )

    # Normalize and combine points
    print("\nNormalizing depths:")
    for view_name, points in [
        ("Front", (front_x, front_y, front_z)),
        ("Back", (back_x, back_y, back_z)),
    ]:
        if len(points[0]) > 0:
            print(
                f"{view_name} z-range: {points[2].min():.3f} to {points[2].max():.3f}"
            )

    # Center and normalize points
    if len(front_x) > 0:
        front_width = np.max(front_x) - np.min(front_x)
        front_x_centered = front_x - np.min(front_x) - front_width / 2
        front_z_norm = 0.5 + (front_z - np.min(front_z)) * depth_threshold / (
            np.max(front_z) - np.min(front_z)
        )

    if len(back_x) > 0:
        back_width = np.max(back_x) - np.min(back_x)
        back_x_centered = -(back_x - np.min(back_x) - back_width / 2)
        back_z_norm = 0.5 - (back_z - np.min(back_z)) * depth_threshold / (
            np.max(back_z) - np.min(back_z)
        )

    # Combine points
    combined_x = np.concatenate([front_x_centered, back_x_centered])
    combined_y = np.concatenate([front_y, back_y])
    combined_z = np.concatenate([front_z_norm, back_z_norm])
    combined_colors = [f'rgb({",".join(c)})' for c in front_colors] + [
        f'rgb({",".join(c)})' for c in back_colors
    ]

    print(f"\nFinal combined point count: {len(combined_x)}")
    print(f"Combined z-range: {combined_z.min():.3f} to {combined_z.max():.3f}")

    # Create combined figure
    combined_fig = go.Figure(
        data=[
            go.Scatter3d(
                x=combined_x,
                y=combined_y,
                z=combined_z,
                mode="markers",
                marker=dict(size=1, color=combined_colors, opacity=1),
                showlegend=False,
            )
        ]
    )

    # Update layout
    combined_fig.update_layout(
        scene=dict(
            aspectratio=dict(x=1, y=1 / (original_width / original_height), z=0.5),
            aspectmode="manual",
            camera=dict(
                eye=dict(x=1.25, y=-0.25, z=1.25),
                up=dict(x=0, y=0, z=0),
                center=dict(x=0, y=0, z=0),
            ),
            xaxis=dict(
                range=[-original_width / 2, original_width / 2],
                showticklabels=False,
                showgrid=False,
                zeroline=False,
                showline=False,
                showbackground=False,
            ),
            yaxis=dict(
                range=[0, original_height],
                showticklabels=False,
                showgrid=False,
                zeroline=False,
                showline=False,
                showbackground=False,
            ),
            zaxis=dict(
                range=[0, 1],
                showticklabels=False,
                showgrid=False,
                zeroline=False,
                showline=False,
                showbackground=False,
            ),
        ),
        uirevision="true",
        showlegend=False,
        margin=dict(l=0, r=0, t=0, b=0, pad=0),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
    )

    print("\n=== Mesh Combination Complete ===")
    return combined_fig

def validate_interpolation(ground_truth, interpolated_points):
    """
    Validate interpolation quality against ground truth
    """
    # Calculate Hausdorff distance
    tree_gt = cKDTree(ground_truth)
    tree_interp = cKDTree(interpolated_points)
    
    # Forward distance
    distances_fw, _ = tree_gt.query(interpolated_points)
    # Backward distance
    distances_bw, _ = tree_interp.query(ground_truth)
    
    # Compute metrics
    hausdorff = max(np.max(distances_fw), np.max(distances_bw))
    mean_error = np.mean(distances_fw)
    
    return {
        'hausdorff_distance': hausdorff,
        'mean_error': mean_error,
        'std_error': np.std(distances_fw)
    }

import numpy as np
from scipy.spatial import cKDTree
from scipy.interpolate import RBFInterpolator

def advanced_surface_interpolation(front_points, back_points, n_samples=50):
    """
    A novel approach to surface interpolation between point clouds using
    local geometric features and radial basis functions.
    
    Args:
        front_points: Points from front view (N x 3 array)
        back_points: Points from back view (M x 3 array)
        n_samples: Number of interpolation points to generate
    """
    # Step 1: Estimate local surface properties
    def estimate_local_curvature(points, k=10):
        # Create KD-tree for efficient neighbor search
        tree = cKDTree(points)
        
        # Find k nearest neighbors for each point
        distances, indices = tree.query(points, k=k)
        
        # Calculate local curvature using PCA
        curvatures = []
        normals = []
        
        for idx_set in indices:
            local_points = points[idx_set]
            centered = local_points - np.mean(local_points, axis=0)
            
            # Compute covariance matrix
            cov = np.dot(centered.T, centered)
            
            # Get eigenvalues and eigenvectors
            eigenvals, eigenvecs = np.linalg.eigh(cov)
            
            # Smallest eigenvalue indicates curvature
            curvatures.append(eigenvals[0] / np.sum(eigenvals))
            # Corresponding eigenvector is the normal
            normals.append(eigenvecs[:, 0])
            
        return np.array(curvatures), np.array(normals)

    # Step 2: Create interpolation points with geometric guidance
    def generate_interpolation_path(p1, p2, n1, n2, curvature, samples):
        """
        Generate interpolation points following estimated surface curvature
        """
        # Create parameter space
        t = np.linspace(0, 1, samples)
        
        # Novel mathematical model for path generation
        # Using cubic Hermite spline with curvature adjustment
        h00 = 2*t**3 - 3*t**2 + 1
        h10 = t**3 - 2*t**2 + t
        h01 = -2*t**3 + 3*t**2
        h11 = t**3 - t**2
        
        # Adjust tangent vectors based on curvature
        alpha = np.exp(-curvature * 5)  # Curvature scaling factor
        
        # Generate interpolated points
        path = (h00[:, np.newaxis] * p1 + 
               h10[:, np.newaxis] * (n1 * alpha) +
               h01[:, np.newaxis] * p2 +
               h11[:, np.newaxis] * (n2 * alpha))
        
        return path

    # Calculate local geometric properties
    front_curvatures, front_normals = estimate_local_curvature(front_points)
    back_curvatures, back_normals = estimate_local_curvature(back_points)
    
    # Match corresponding points between views
    def find_correspondences(front, back, front_normals, back_normals):
        """
        Find corresponding points between views using geometric similarity
        """
        tree = cKDTree(back)
        _, indices = tree.query(front, k=1)
        
        # Calculate normal compatibility
        normal_dot = np.abs(np.sum(front_normals * back_normals[indices], axis=1))
        
        # Filter based on normal compatibility
        valid = normal_dot > 0.7
        
        return valid, indices[valid]

    valid_mask, correspondences = find_correspondences(
        front_points, back_points, front_normals, back_normals)
    
    # Generate interpolated surfaces
    interpolated_points = []
    for i, (is_valid, corr_idx) in enumerate(zip(valid_mask, correspondences)):
        if is_valid:
            path = generate_interpolation_path(
                front_points[i], 
                back_points[corr_idx],
                front_normals[i],
                back_normals[corr_idx],
                (front_curvatures[i] + back_curvatures[corr_idx]) / 2,
                n_samples
            )
            interpolated_points.append(path)
    
    return np.vstack(interpolated_points)

def combine_views_with_gap_filling_curvature(combined_fig, point_spacing=2):
    """
    Creates an enhanced 3D visualization with curvature-based gap filling.
    
    Args:
        combined_fig: Original combined figure with front and back views
        point_spacing: Controls density of interpolated points
        
    Returns:
        plotly.graph_objects.Figure: New figure with interpolated points
    """
    import plotly.graph_objects as go
    
    # Extract points and colors from original figure
    points = np.column_stack([
        combined_fig.data[0].x,
        combined_fig.data[0].y,
        combined_fig.data[0].z
    ])
    colors = np.array([list(map(int, c.strip('rgb()').split(','))) 
                      for c in combined_fig.data[0].marker.color])
    
    # Split into front and back points
    split_x = np.median(points[:, 0])
    front_mask = points[:, 0] > split_x
    back_mask = ~front_mask
    
    # Get interpolated points using our advanced method
    interpolated_points = advanced_surface_interpolation(
        points[front_mask],
        points[back_mask]
    )
    
    # Create interpolated colors
    # We'll use a gradient between corresponding front and back colors
    interpolated_colors = []
    for i in range(len(interpolated_points)):
        front_color = colors[front_mask][i % len(colors[front_mask])]
        back_color = colors[back_mask][i % len(colors[back_mask])]
        color_steps = np.linspace(front_color, back_color, 
                                interpolated_points.shape[0])
        interpolated_colors.extend([f'rgb({int(r)},{int(g)},{int(b)})' 
                                  for r, g, b in color_steps])
    
    # Create new figure with original and interpolated points
    filled_fig = go.Figure(data=[
        # Original points
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode='markers',
            marker=dict(
                size=combined_fig.data[0].marker.size,
                color=[f'rgb({r},{g},{b})' for r,g,b in colors],
                opacity=combined_fig.data[0].marker.opacity
            ),
            name='Original Points'
        ),
        # Interpolated points
        go.Scatter3d(
            x=interpolated_points[:, 0],
            y=interpolated_points[:, 1],
            z=interpolated_points[:, 2],
            mode='markers',
            marker=dict(
                size=combined_fig.data[0].marker.size * 0.8,  # Slightly smaller
                color=interpolated_colors,
                opacity=0.8  # Slightly more transparent
            ),
            name='Interpolated Points'
        )
    ])
    
    # Copy layout from original figure
    filled_fig.update_layout(combined_fig.layout)
    
    return filled_fig

def process_orthogonal_views(
    front,
    back=None,
    top=None,
    target_size=128,
    spatial_strictness=0.5,
    color_threshold=0.99,
):
    """
    Process orthogonal views with normalized image sizes.
    """
    # Process all three views
    image_paths = {"front": front, "back": back, "top": top}

    try:
        # First normalize all images
        normalized_images, _ = load_and_normalize_images(
            image_paths, target_size=target_size
        )

        # Create meshes for each view
        front_fig = create_object_mesh(
            normalized_images["front"], depth_threshold=0.1, target_size=target_size
        )
        front_fig = remove_outliers(
            front_fig,
            spatial_strictness=spatial_strictness,
            color_threshold=color_threshold,
        )

        back_fig = None
        if back is not None:
            back_fig = create_object_mesh(
                normalized_images["back"], depth_threshold=0.1, target_size=target_size
            )
            back_fig = remove_outliers(
                back_fig,
                spatial_strictness=spatial_strictness,
                color_threshold=color_threshold,
            )

        # top_fig = None
        # if back is not None:
        #     top_fig = create_object_mesh(normalized_images['top'],  target_size=target_size)
        #     top_fig = remove_outliers(top_fig, spatial_strictness=0.5, color_threshold=0.05)

        # Combine meshes
        combined_mesh = combine_front_back_meshes(front_fig, back_fig)
        # combined_mesh.show()
        filled_mesh = combine_views_with_gap_filling(
            combined_mesh,
            point_spacing=2
        )
        

        # filled_mesh.show()
        # validation = validate_interpolation(filled_mesh, filled_mesh)
        # print(f"Validation result: {validation}")

        return filled_mesh.to_dict()

    except Exception as e:
        print(f"Error during processing: {str(e)}")
        return None


orthogonal_views = process_orthogonal_views(
    front="../static/uploads/Chair/front.jpg",
    back="../static/uploads/Chair/back.jpg",
    top="../static/uploads/Chair/top.jpg",
    target_size=128,
)


Loading and normalizing images...
Front view background color (RGB): (255, 255, 255)
Front view normalized size: 128x128
Back view background color (RGB): (255, 255, 255)
Back view normalized size: 128x128
Top view background color (RGB): (255, 255, 255)
Top view normalized size: 128x128
Loading image...
It's already a numpy array - verify format
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...

After background removal:
Result RGB shape: (128, 128, 3)
Alpha mask shape: (128, 128)
Non-zero alpha pixels: 1202


c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.




Depth map statistics:
Depth map shape: (128, 128)
Depth range: 16.000 to 255.000

After resizing:
Resized depth map shape: (128, 128)
Valid mask points: 1202

Normalized depth range: 0.000 to 1.000

Point generation results:
Total possible points: 16384
Points after masking: 1202
Final points after depth threshold: 1201

=== Mesh Creation Complete ===
Outliers removed: 12
Loading image...
It's already a numpy array - verify format
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...

After background removal:
Result RGB shape: (128, 128, 3)
Alpha mask shape: (128, 128)
Non-zero alpha pixels: 1181


c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.




Depth map statistics:
Depth map shape: (128, 128)
Depth range: 18.000 to 255.000

After resizing:
Resized depth map shape: (128, 128)
Valid mask points: 1181

Normalized depth range: 0.000 to 1.000

Point generation results:
Total possible points: 16384
Points after masking: 1181
Final points after depth threshold: 1176

=== Mesh Creation Complete ===
Outliers removed: 13

=== Starting Mesh Combination ===

Front view initial points: 1189

Back view initial points: 1163

Sampling Front view:
Original points: 1189
Target points: 1189

Sampling Back view:
Original points: 1163
Target points: 1163

Dimensions:
Original width: 80.0
Original height: 87.0
Aspect ratio: 0.9195402298850575

Normalizing depths:
Front z-range: 0.100 to 0.314
Back z-range: 0.101 to 0.308

Final combined point count: 2352
Combined z-range: 0.250 to 0.750

=== Mesh Combination Complete ===

Fill Results:
Front fill points: 10442
Back fill points: 11234


In [7]:
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
# import numpy as np
# from PIL import Image
# import io

# def calculate_model_size(x, y, z, zoom_factor=0.5):
#     """
#     Calculates the appropriate camera distance based on the model's dimensions.
    
#     Parameters:
#     -----------
#     x, y, z : array-like
#         Coordinates of the model points
#     zoom_factor : float
#         Controls the zoom level:
#         - Values less than 1 zoom in (e.g., 0.5 shows the model twice as large)
#         - Values greater than 1 zoom out (e.g., 2.0 shows the model half as large)
#         - A value of 1.0 uses the default zoom level
    
#     Returns:
#     --------
#     float
#         Calculated camera distance
#     """
#     # Calculate the model's bounding box dimensions
#     x_range = max(x) - min(x)
#     y_range = max(y) - min(y)
#     z_range = max(z) - min(z)
    
#     # Find the largest dimension to ensure the whole model is visible
#     max_dimension = max(x_range, y_range, z_range)
    
#     # Apply zoom factor to adjust the view distance
#     # We multiply by 1.5 as a base factor to ensure some padding around the model
#     return max_dimension * 1.5 * zoom_factor

# def create_orthographic_views(view_dict, zoom_factor=0.5, width=1500, height=1500, background_color='white'):
#     """
#     Creates a grid of orthographic views with adjustable zoom level.
    
#     Parameters:
#     -----------
#     view_dict : dict
#         Dictionary containing the mesh data
#     zoom_factor : float
#         Controls the zoom level:
#         - Values less than 1 zoom in (e.g., 0.5 shows the model twice as large)
#         - Values greater than 1 zoom out (e.g., 2.0 shows the model half as large)
#     width, height : int
#         Dimensions of the output image
#     background_color : str
#         Background color for the plots
#     """
#     # Create subplot grid
#     fig = make_subplots(
#         rows=2, cols=3,
#         specs=[[{'type': 'scene'} for _ in range(3)] for _ in range(2)],
#         subplot_titles=('Front View', 'Side View', 'Back View',
#                        'Top View', 'Bottom View', 'Isometric View')
#     )
    
#     # Extract trace data
#     trace_data = view_dict.get('data', [{}])[0] if isinstance(view_dict.get('data'), list) else {}
#     x_data = trace_data.get('x', [])
#     y_data = trace_data.get('y', [])
#     z_data = trace_data.get('z', [])
    
#     # Calculate camera distance with zoom factor
#     camera_distance = calculate_model_size(x_data, y_data, z_data, zoom_factor)
    
#     # Define camera positions with adjusted distances
#     views = {
#         (1, 1): dict(  # Front
#             up=dict(x=0, y=-1, z=0),
#             center=dict(x=0, y=0, z=0),
#             eye=dict(x=0, y=0, z=camera_distance)
#         ),
#         (1, 2): dict(  # Side
#             up=dict(x=0, y=-1, z=0),
#             center=dict(x=0, y=0, z=0),
#             eye=dict(x=camera_distance, y=0, z=0)
#         ),
#         (1, 3): dict(  # Back
#             up=dict(x=0, y=-1, z=0),
#             center=dict(x=0, y=0, z=0),
#             eye=dict(x=0, y=0, z=-camera_distance)
#         ),
#         (2, 1): dict(  # Top
#             up=dict(x=0, y=0, z=-1),
#             center=dict(x=0, y=0, z=0),
#             eye=dict(x=0, y=-camera_distance, z=0)
#         ),
#         (2, 2): dict(  # Bottom
#             up=dict(x=0, y=0, z=1),
#             center=dict(x=0, y=0, z=0),
#             eye=dict(x=0, y=camera_distance, z=0)
#         ),
#         (2, 3): dict(  # Isometric
#             up=dict(x=0, y=0, z=1),
#             center=dict(x=0, y=0, z=0),
#             eye=dict(
#                 x=camera_distance * 0.7,
#                 y=camera_distance * 0.7,
#                 z=camera_distance * 0.7
#             )
#         )
#     }
    
#     # Add traces to each subplot
#     for pos, camera in views.items():
#         row, col = pos
        
#         fig.add_trace(
#             go.Scatter3d(
#                 x=x_data,
#                 y=y_data,
#                 z=z_data,
#                 mode=trace_data.get('mode', 'markers'),
#                 marker=trace_data.get('marker', dict())
#             ),
#             row=row,
#             col=col
#         )
        
#         # Update scene properties
#         fig.update_scenes(
#             camera=camera,
#             xaxis=dict(visible=False, showbackground=False),
#             yaxis=dict(visible=False, showbackground=False),
#             zaxis=dict(visible=False, showbackground=False),
#             aspectmode='data',
#             row=row,
#             col=col
#         )
    
#     # Update overall layout
#     fig.update_layout(
#         paper_bgcolor=background_color,
#         plot_bgcolor=background_color,
#         margin=dict(l=0, r=0, t=30, b=0),
#         showlegend=False,
#         width=width,
#         height=height
#     )
    
#     return fig

# def extract_orthographic_view(view_dict, view_type='top', zoom_factor=0.5, width=800, height=800, background_color='white'):
#     """
#     Extracts a single orthographic view with adjustable zoom level.
    
#     Parameters:
#     -----------
#     view_dict : dict
#         Dictionary containing the mesh data
#     view_type : str
#         Type of view to extract ('top', 'front', 'side', 'back', 'isometric')
#     zoom_factor : float
#         Controls the zoom level (less than 1 zooms in, greater than 1 zooms out)
#     width, height : int
#         Dimensions of the output image
#     background_color : str
#         Background color for the plot
#     """
#     # Extract trace data
#     trace_data = view_dict.get('data', [{}])[0] if isinstance(view_dict.get('data'), list) else {}
#     x_data = trace_data.get('x', [])
#     y_data = trace_data.get('y', [])
#     z_data = trace_data.get('z', [])
    
#     # Calculate camera distance with zoom factor
#     camera_distance = calculate_model_size(x_data, y_data, z_data, zoom_factor)
    
#     # Create camera presets with adjusted distance
#     camera_presets = {
#         'front': dict(
#             up=dict(x=0, y=-1, z=0),
#             center=dict(x=0, y=0, z=0),
#             eye=dict(x=0, y=0, z=camera_distance)
#         ),
#         'top': dict(
#             up=dict(x=0.5, y=0.0, z=-1.5),
#             center=dict(x=0.5, y=0, z=0.5),
#             eye=dict(x=0.0, y=-camera_distance, z=0.0)
#         ),
#         'side': dict(
#             up=dict(x=0, y=-1, z=0),
#             center=dict(x=0, y=0, z=0),
#             eye=dict(x=camera_distance, y=0, z=0)
#         ),
#         'back': dict(
#             up=dict(x=0, y=-1, z=0),
#             center=dict(x=0, y=0, z=0),
#             eye=dict(x=0, y=0, z=-camera_distance)
#         ),
#         'isometric': dict(
#             up=dict(x=0, y=0, z=1),
#             center=dict(x=0, y=0, z=0),
#             eye=dict(
#                 x=camera_distance * 0.7,
#                 y=camera_distance * 0.7,
#                 z=camera_distance * 0.7
#             )
#         )
#     }
    
#     # Create figure
#     fig = go.Figure()
    
#     # Add trace
#     fig.add_trace(
#         go.Scatter3d(
#             x=x_data,
#             y=y_data,
#             z=z_data,
#             mode=trace_data.get('mode', 'markers'),
#             marker=trace_data.get('marker', dict())
#         )
#     )
    
#     # Update layout
#     fig.update_layout(
#         scene_camera=camera_presets.get(view_type, camera_presets['front']),
#         scene=dict(
#             xaxis=dict(visible=False, showbackground=False),
#             yaxis=dict(visible=False, showbackground=False),
#             zaxis=dict(visible=False, showbackground=False),
#             aspectmode='data'
#         ),
#         paper_bgcolor=background_color,
#         plot_bgcolor=background_color,
#         margin=dict(l=0, r=0, t=0, b=0),
#         showlegend=False,
#         width=width,
#         height=height
#     )
    
#     # Convert to image
#     img_bytes = fig.to_image(format="png", width=width, height=height)
#     img = Image.open(io.BytesIO(img_bytes))
    
#     return img


# # # For all views in a grid with zoomed in view (e.g., zoom_factor=0.3 for closer view)
# # grid_fig = create_orthographic_views(orthogonal_views, zoom_factor=0.1)
# # grid_fig.write_image("all_views.png")

# # # For a single view with custom zoom
# # top_view_img = extract_orthographic_view(
# #     orthogonal_views,
# #     view_type='top',
# #     zoom_factor=0.1  # Adjust this value to control zoom level
# # )
# # top_view_img.save('top_view.png')

In [25]:
import os
import json
import numpy as np
import cv2
import time
import psutil
from pathlib import Path
from datetime import datetime
from scipy.spatial import cKDTree
from functools import wraps
import plotly.utils
import traceback

def timer_decorator(func):
    """Accurately measures function execution time and resource usage."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_cpu = time.process_time()
        start_wall = time.perf_counter()
        initial_memory = psutil.Process().memory_info().rss / 1024 / 1024
        
        result = func(*args, **kwargs)
        
        end_cpu = time.process_time()
        end_wall = time.perf_counter()
        final_memory = psutil.Process().memory_info().rss / 1024 / 1024
        
        metrics = {
            'cpu_time_seconds': end_cpu - start_cpu,
            'wall_time_seconds': end_wall - start_wall,
            'memory_used_mb': final_memory - initial_memory,
            'peak_memory_mb': final_memory
        }
        
        if isinstance(result, dict):
            result['performance_metrics'] = metrics
        return result
    return wrapper

def calculate_point_cloud_quality(points, colors):
    """Evaluates the quality of point cloud reconstruction."""
    if len(points) == 0:
        return None
        
    # Create KD-tree for efficient neighbor searches
    tree = cKDTree(points)
    
    # Calculate density uniformity
    distances, _ = tree.query(points, k=2)
    nearest_distances = distances[:, 1]
    mean_distance = np.mean(nearest_distances)
    density_uniformity = 1.0 - (np.std(nearest_distances) / mean_distance) if mean_distance > 0 else 0.0
    
    # Calculate color consistency
    k = min(10, len(points))
    _, indices = tree.query(points, k=k)
    color_variations = []
    
    for idx_group in indices:
        neighborhood_colors = colors[idx_group]
        if len(neighborhood_colors) > 0:
            color_var = np.mean(np.std(neighborhood_colors, axis=0))
            color_variations.append(color_var)
    
    color_consistency = 1.0 - (np.mean(color_variations) / 255.0) if color_variations else 0.0
    
    # Calculate spatial metrics
    extents = np.ptp(points, axis=0)
    volume = np.prod(extents) if np.all(extents > 0) else 0
    point_density = len(points) / volume if volume > 0 else 0
    
    # Calculate spatial uniformity
    centroid = np.mean(points, axis=0)
    distances_to_centroid = np.linalg.norm(points - centroid, axis=1)
    mean_centroid_distance = np.mean(distances_to_centroid)
    spatial_uniformity = 1.0 - (np.std(distances_to_centroid) / mean_centroid_distance) if mean_centroid_distance > 0 else 0.0
    
    return {
        'density_uniformity': float(density_uniformity),
        'color_consistency': float(color_consistency),
        'point_density': float(point_density),
        'spatial_uniformity': float(spatial_uniformity),
        'total_points': len(points),
        'coverage_volume': float(volume)
    }

def create_top_mask_from_mesh(points, resolution=256):
    """Creates a top-view binary mask from 3D points."""
    grid = np.zeros((resolution, resolution))
    
    # Normalize coordinates to grid space
    x_min, x_max = points[:,0].min(), points[:,0].max()
    y_min, y_max = points[:,1].min(), points[:,1].max()
    
    x_scaled = ((points[:,0] - x_min) / (x_max - x_min) * (resolution-1)).astype(int)
    y_scaled = ((points[:,1] - y_min) / (y_max - y_min) * (resolution-1)).astype(int)
    
    # Use KD-tree for efficient point density calculation
    tree = cKDTree(np.column_stack([x_scaled, y_scaled]))
    
    for i in range(resolution):
        for j in range(resolution):
            indices = tree.query_ball_point([i, j], r=2)
            if indices:
                z_values = points[indices, 2]
                grid[j,i] = len(indices) * np.max(z_values)
    
    # Normalize and threshold
    if grid.max() > 0:
        grid = grid / grid.max()
    mask = (grid > 0.1).astype(np.uint8)
    
    return mask

def evaluate_depth_accuracy(points, top_image, resolution=256):
    """Evaluates reconstruction accuracy using top view comparison."""
    # Generate masks
    predicted_mask = create_top_mask_from_mesh(points, resolution)
    
    # Process top view image
    if len(top_image.shape) == 3:
        top_gray = cv2.cvtColor(top_image, cv2.COLOR_BGR2GRAY)
    else:
        top_gray = top_image
    
    top_gray = cv2.resize(top_gray, (resolution, resolution))
    _, ground_truth_mask = cv2.threshold(top_gray, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Calculate overlap metrics
    intersection = np.logical_and(predicted_mask, ground_truth_mask)
    union = np.logical_or(predicted_mask, ground_truth_mask)
    
    intersection_sum = np.sum(intersection)
    union_sum = np.sum(union)
    predicted_sum = np.sum(predicted_mask)
    ground_truth_sum = np.sum(ground_truth_mask)
    
    # Calculate accuracy metrics
    iou = intersection_sum / union_sum if union_sum > 0 else 0
    precision = intersection_sum / predicted_sum if predicted_sum > 0 else 0
    recall = intersection_sum / ground_truth_sum if ground_truth_sum > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    # Calculate contour metrics
    contour_metrics = calculate_contour_metrics(predicted_mask, ground_truth_mask)
    
    return {
        'silhouette_metrics': {
            'iou_score': float(iou),
            'precision': float(precision),
            'recall': float(recall),
            'f1_score': float(f1_score)
        },
        'contour_metrics': contour_metrics
    }

def calculate_contour_metrics(predicted_mask, ground_truth_mask):
    """Calculates shape similarity using contour analysis."""
    pred_uint8 = (predicted_mask * 255).astype(np.uint8)
    gt_uint8 = (ground_truth_mask * 255).astype(np.uint8)
    
    # Find contours
    pred_contours, _ = cv2.findContours(pred_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    gt_contours, _ = cv2.findContours(gt_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not pred_contours or not gt_contours:
        return {'shape_similarity': 0.0, 'boundary_accuracy': 0.0}
    
    # Get largest contours
    pred_contour = max(pred_contours, key=cv2.contourArea)
    gt_contour = max(gt_contours, key=cv2.contourArea)
    
    # Calculate shape matching score
    match_score = cv2.matchShapes(pred_contour, gt_contour, cv2.CONTOURS_MATCH_I2, 0.0)
    
    # Calculate boundary accuracy
    pred_points = pred_contour.squeeze()
    gt_points = gt_contour.squeeze()
    
    if len(pred_points.shape) > 1 and len(gt_points.shape) > 1:
        distances = np.max([
            np.min([np.linalg.norm(p1 - p2) for p2 in gt_points]) 
            for p1 in pred_points
        ])
        boundary_accuracy = 1.0 / (1.0 + distances)
    else:
        boundary_accuracy = 0.0
    
    return {
        'shape_similarity': float(1 / (1 + match_score)),
        'boundary_accuracy': float(boundary_accuracy)
    }

def calculate_image_metrics(image_data, image_paths):
    """
    Calculate metrics for input images with improved error handling.
    
    Args:
        image_data: Dictionary containing the loaded images, keyed by view name ('front', 'back', 'top')
        image_paths: Dictionary containing paths for each view
    
    Returns:
        Dictionary containing metrics for each successfully processed image
    """
    metrics = {}
    
    # Process each view that has both image data and a valid path
    for view_name, img in image_data.items():
        try:
            # Get the corresponding path for this view
            path = image_paths.get(view_name)
            
            if img is not None and path is not None:
                # Calculate metrics for this view
                metrics[view_name] = {
                    'resolution': list(img.shape[:2]),  # Convert to list for JSON serialization
                    'aspect_ratio': float(img.shape[1] / img.shape[0]),
                    'mean_brightness': float(np.mean(img)),
                    'contrast': float(np.std(img)),
                    'file_size_kb': float(os.path.getsize(str(path)) / 1024) if Path(path).exists() else 0
                }
        except Exception as e:
            print(f"Warning: Failed to calculate metrics for {view_name}: {str(e)}")
            continue
    
    return metrics

@timer_decorator
def calculate_reconstruction_metrics(reconstruction_results, images=None):
    """Calculates comprehensive reconstruction quality metrics."""
    if reconstruction_results is None or 'data' not in reconstruction_results:
        return None

    try:
        # Extract point cloud data
        points = np.array([
            reconstruction_results['data'][0]['x'],
            reconstruction_results['data'][0]['y'],
            reconstruction_results['data'][0]['z']
        ]).T
        
        # Extract colors
        colors = np.array([
            [int(c.strip('rgb()').split(',')[i]) for i in range(3)]
            for c in reconstruction_results['data'][0]['marker']['color']
        ])
        
        # Calculate all metrics
        quality_metrics = calculate_point_cloud_quality(points, colors)
        depth_metrics = None
        if images is not None and 'top' in images and images['top'] is not None:
            depth_metrics = evaluate_depth_accuracy(points, images['top'])
        
        metrics = {
            'quality_metrics': quality_metrics,
            'depth_metrics': depth_metrics,
            'bounding_box': {
                'x': [float(np.min(points[:,0])), float(np.max(points[:,0]))],
                'y': [float(np.min(points[:,1])), float(np.max(points[:,1]))],
                'z': [float(np.min(points[:,2])), float(np.max(points[:,2]))]
            }
        }
        
        return metrics
        
    except Exception as e:
        print(f"Error in reconstruction metrics calculation: {str(e)}")
        return None

def load_product_images(product_dir):
    """Loads and validates product images."""
    images = {}
    required_views = ['front.jpg', 'back.jpg']
    optional_views = ['top.jpg']

    # Load required views
    for view in required_views:
        path = product_dir / view
        if not path.exists():
            print(f"Missing required view: {view}")
            return None
        images[view.split('.')[0]] = cv2.imread(str(path))

    # Load optional views
    for view in optional_views:
        path = product_dir / view
        if path.exists():
            images[view.split('.')[0]] = cv2.imread(str(path))

    return images

def write_summary(f, metrics):
    """
    Write a human-readable summary of the metrics.
    
    Args:
        f: File object to write to
        metrics: Dictionary of metrics to summarize
    """
    f.write("=== 3D Reconstruction Evaluation Summary ===\n\n")
    
    # Write image metrics
    if metrics.get('image_metrics'):
        f.write("Input Image Quality:\n")
        for view, img_metrics in metrics['image_metrics'].items():
            f.write(f"\n{view.capitalize()} View:\n")
            f.write(f"- Resolution: {img_metrics['resolution']}\n")
            f.write(f"- Aspect Ratio: {img_metrics['aspect_ratio']:.2f}\n")
            f.write(f"- Mean Brightness: {img_metrics['mean_brightness']:.1f}\n")
            f.write(f"- Contrast: {img_metrics['contrast']:.1f}\n")
        f.write("\n")
    
    # Write reconstruction metrics
    if metrics.get('reconstruction_metrics'):
        rec_metrics = metrics['reconstruction_metrics']
        if rec_metrics.get('quality_metrics'):
            quality = rec_metrics['quality_metrics']
            f.write("\nReconstruction Quality:\n")
            for key, value in quality.items():
                f.write(f"- {key}: {value}\n")
        
        if rec_metrics.get('depth_metrics'):
            depth = rec_metrics['depth_metrics']
            f.write("\nDepth Metrics:\n")
            for key, value in depth.items():
                f.write(f"- {key}: {value}\n")
    
    # Write performance metrics
    if metrics.get('performance_metrics'):
        f.write("\nPerformance Metrics:\n")
        for key, value in metrics['performance_metrics'].items():
            f.write(f"- {key}: {value}\n")
            
@timer_decorator
def process_single_product(front_path, back_path, top_path=None, product_dir=None):
    """
    Process a single product with comprehensive error handling and data validation.
    
    Args:
        front_path: Path to front view image
        back_path: Path to back view image
        top_path: Optional path to top view image
        product_dir: Directory for saving results
    """
    try:
        # Convert paths to Path objects
        front_path = Path(front_path)
        back_path = Path(back_path)
        top_path = Path(top_path) if top_path else None
        product_dir = Path(product_dir) if product_dir else None

        # Initialize data structures for images and paths
        image_data = {}
        image_paths = {
            'front': front_path,
            'back': back_path,
            'top': top_path
        }
        
        # Load each image with validation
        for view, path in image_paths.items():
            if path and path.exists():
                try:
                    img = cv2.imread(str(path))
                    if img is not None:
                        image_data[view] = img
                except Exception as e:
                    print(f"Warning: Failed to load {view} image: {str(e)}")
                    continue

        # Generate reconstruction
        reconstruction_results = process_orthogonal_views(
            front=str(front_path),
            back=str(back_path),
            top=str(top_path) if top_path else None,
            target_size=128
        )

        # Save reconstruction results
        if reconstruction_results and product_dir:
            plot_file = product_dir / 'plot_data.json'
            plot_file.parent.mkdir(parents=True, exist_ok=True)
            with open(plot_file, 'w') as f:
                json.dump(reconstruction_results, cls=plotly.utils.PlotlyJSONEncoder, fp=f)

        # Calculate and compile all metrics
        metrics = {
            'timestamp': datetime.now().isoformat(),
            'image_metrics': calculate_image_metrics(image_data, image_paths),
            'reconstruction_metrics': calculate_reconstruction_metrics(
                reconstruction_results,
                image_data
            ) if reconstruction_results else None
        }

        # Save results if directory is provided
        if product_dir:
            metrics_file = product_dir / 'evaluation_metrics.json'
            with open(metrics_file, 'w') as f:
                json.dump(metrics, cls=plotly.utils.PlotlyJSONEncoder, fp=f)
            
            # Create human-readable summary
            summary_file = product_dir / 'evaluation_summary.txt'
            with open(summary_file, 'w') as f:
                write_summary(f, metrics)

        return metrics

    except Exception as e:
        print(f"Error in process_single_product: {str(e)}")
        import traceback
        traceback.print_exc()  # Print full error traceback for debugging
        return None

def save_evaluation_results(metrics, product_dir):
    """
    Saves evaluation results to files with both detailed metrics and human-readable summary.
    Creates both JSON and text-based reports for easy analysis.
    """
    output_dir = Path(product_dir)
    metrics_file = output_dir / 'evaluation_metrics.json'
    
    # Save detailed metrics in JSON format for programmatic access
    with open(metrics_file, 'w') as f:
        json.dump(metrics, f, indent=2)
    
    # Create a human-readable summary report
    summary_file = output_dir / 'evaluation_summary.txt'
    with open(summary_file, 'w') as f:
        f.write("=== 3D Reconstruction Evaluation Summary ===\n\n")
        
        # Write image metrics
        if metrics.get('image_metrics'):
            f.write("Input Image Quality:\n")
            for view, img_metrics in metrics['image_metrics'].items():
                f.write(f"\n{view.capitalize()} View:\n")
                f.write(f"- Resolution: {img_metrics['resolution']}\n")
                f.write(f"- Aspect Ratio: {img_metrics['aspect_ratio']:.2f}\n")
                f.write(f"- Mean Brightness: {img_metrics['mean_brightness']:.1f}\n")
                f.write(f"- Contrast: {img_metrics['contrast']:.1f}\n")
            f.write("\n")
        
        # Write reconstruction quality metrics
        if metrics.get('reconstruction_metrics'):
            quality = metrics['reconstruction_metrics'].get('quality_metrics', {})
            f.write("\nReconstruction Quality:\n")
            f.write(f"- Total Points: {quality.get('total_points', 0):,}\n")
            f.write(f"- Density Uniformity: {quality.get('density_uniformity', 0):.3f}\n")
            f.write(f"- Color Consistency: {quality.get('color_consistency', 0):.3f}\n")
            f.write(f"- Spatial Uniformity: {quality.get('spatial_uniformity', 0):.3f}\n")
            f.write(f"- Point Density: {quality.get('point_density', 0):.1f} points/unit³\n")
            f.write(f"- Coverage Volume: {quality.get('coverage_volume', 0):.1f} units³\n\n")
            
            # Write depth evaluation metrics if available
            if metrics['reconstruction_metrics'].get('depth_metrics'):
                depth = metrics['reconstruction_metrics']['depth_metrics']
                f.write("Depth Accuracy Metrics:\n")
                f.write(f"- IoU Score: {depth['silhouette_metrics']['iou_score']:.3f}\n")
                f.write(f"- Precision: {depth['silhouette_metrics']['precision']:.3f}\n")
                f.write(f"- Recall: {depth['silhouette_metrics']['recall']:.3f}\n")
                f.write(f"- F1 Score: {depth['silhouette_metrics']['f1_score']:.3f}\n")
                f.write(f"- Shape Similarity: {depth['contour_metrics']['shape_similarity']:.3f}\n")
                f.write(f"- Boundary Accuracy: {depth['contour_metrics']['boundary_accuracy']:.3f}\n\n")
        
        # Write performance metrics
        if metrics.get('performance_metrics'):
            perf = metrics['performance_metrics']
            f.write("Performance Metrics:\n")
            f.write(f"- Processing Time: {perf['wall_time_seconds']:.2f} seconds\n")
            f.write(f"- CPU Time: {perf['cpu_time_seconds']:.2f} seconds\n")
            f.write(f"- Memory Used: {perf['memory_used_mb']:.1f} MB\n")
            f.write(f"- Peak Memory: {perf['peak_memory_mb']:.1f} MB\n")


def generate_dataset_summary(results, output_path, product_metadata):
    """
    Generates a comprehensive summary of all processed products, grouped by category.
    
    Args:
        results: Dictionary mapping product IDs to their reconstruction results
        output_path: Path where summary files will be saved
        product_metadata: Dictionary mapping product IDs to their metadata including category
    """
    # First, let's ensure our results are in the correct format
    if not isinstance(results, dict):
        print("Warning: Results must be a dictionary. Converting to appropriate format...")
        if isinstance(results, list):
            results = {str(i): result for i, result in enumerate(results)}
    
    # Group results by category
    category_results = {}
    for product_id, result in results.items():
        # Safely get category with a default value
        if product_id in product_metadata:
            category = product_metadata[product_id]['category']
        else:
            category = 'uncategorized'
            print(f"Warning: No category found for product {product_id}")
        
        # Initialize category list if needed
        if category not in category_results:
            category_results[category] = []
        
        # Add result to appropriate category
        category_results[category].append(result)
    
    # Prepare summary structure
    summary = {
        'total_products': len(results),
        'categories': list(category_results.keys()),
        'timestamp': datetime.now().isoformat(),
        'overall_averages': calculate_dataset_averages(results),
        'overall_distributions': calculate_metric_distributions(results),
        'category_analysis': {}
    }
    
    # Calculate per-category metrics
    for category, cat_results in category_results.items():
        # Convert list of results to dictionary format for our existing functions
        cat_results_dict = {str(i): result for i, result in enumerate(cat_results)}
        
        summary['category_analysis'][category] = {
            'product_count': len(cat_results),
            'averages': calculate_dataset_averages(cat_results_dict),
            'distributions': calculate_metric_distributions(cat_results_dict)
        }
    
    # Create output directory if it doesn't exist
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Save detailed summary as JSON
    print("saving result..")
    with open(output_path / 'dataset_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

    
    
    # Create human-readable report
    with open(output_path / 'dataset_analysis.txt', 'w') as f:
        f.write("=== Dataset Analysis Report ===\n\n")
        f.write(f"Total Products Processed: {summary['total_products']}\n")
        f.write(f"Analysis Date: {summary['timestamp']}\n")
        f.write(f"Product Categories: {', '.join(summary['categories'])}\n\n")
        
        # Overall metrics
        f.write("=== Overall Dataset Metrics ===\n")
        write_metric_section(f, summary['overall_averages'], summary['overall_distributions'])
        
        # Per-category metrics
        f.write("\n=== Category-Specific Analysis ===\n")
        for category, analysis in summary['category_analysis'].items():
            f.write(f"\n{category.upper()} Category (Products: {analysis['product_count']})\n")
            write_metric_section(f, analysis['averages'], analysis['distributions'])

def write_metric_section(f, averages, distributions):
    """Helper function to write metric sections in a consistent format."""
    f.write("\nAverage Metrics:\n")
    for metric, value in averages.items():
        f.write(f"- {metric}: {value:.3f}\n")
    
    f.write("\nMetric Distributions:\n")
    for metric, dist in distributions.items():
        f.write(f"\n{metric}:\n")
        f.write(f"  Min: {dist['min']:.3f}\n")
        f.write(f"  Max: {dist['max']:.3f}\n")
        f.write(f"  Mean: {dist['mean']:.3f}\n")
        f.write(f"  Std Dev: {dist['std']:.3f}\n")

def calculate_dataset_averages(results):
    """
    Calculates average values for key metrics across all products with robust error handling
    and data structure validation.
    
    Args:
        results: Dictionary or list containing reconstruction results
        
    Returns:
        Dictionary of averaged metrics with proper error handling
    """
    # Initialize counters for each metric we want to track
    metrics_sum = {
        'density_uniformity': 0.0,
        'color_consistency': 0.0,
        'spatial_uniformity': 0.0,
        'iou_score': 0.0,
        'shape_similarity': 0.0
    }
    counts = {metric: 0 for metric in metrics_sum}
    
    # Handle both dictionary and list inputs
    results_items = results.items() if isinstance(results, dict) else enumerate(results)
    
    for _, product_metrics in results_items:
        try:
            # Safely navigate the metrics structure
            if isinstance(product_metrics, dict) and 'reconstruction_metrics' in product_metrics:
                recon = product_metrics['reconstruction_metrics']
                
                # Quality metrics
                if recon and isinstance(recon, dict) and 'quality_metrics' in recon:
                    quality = recon['quality_metrics']
                    if quality and isinstance(quality, dict):
                        for metric in ['density_uniformity', 'color_consistency', 'spatial_uniformity']:
                            if metric in quality and quality[metric] is not None:
                                metrics_sum[metric] += float(quality[metric])
                                counts[metric] += 1
                
                # Depth metrics
                if recon and isinstance(recon, dict) and 'depth_metrics' in recon:
                    depth = recon['depth_metrics']
                    if depth and isinstance(depth, dict):
                        if 'silhouette_metrics' in depth:
                            sil = depth['silhouette_metrics']
                            if 'iou_score' in sil:
                                metrics_sum['iou_score'] += float(sil['iou_score'])
                                counts['iou_score'] += 1
                                
                        if 'contour_metrics' in depth:
                            cont = depth['contour_metrics']
                            if 'shape_similarity' in cont:
                                metrics_sum['shape_similarity'] += float(cont['shape_similarity'])
                                counts['shape_similarity'] += 1
                                
        except Exception as e:
            print(f"Warning: Error processing metrics: {str(e)}")
            continue
    
    # Calculate averages with safety checks
    averages = {}
    for metric in metrics_sum:
        if counts[metric] > 0:
            averages[metric] = metrics_sum[metric] / counts[metric]
        else:
            averages[metric] = 0.0
            
    return averages

def calculate_metric_distributions(results):
    """
    Calculates statistical distributions for key metrics with improved error handling
    and data structure validation.
    
    Args:
        results: Dictionary or list containing reconstruction results
        
    Returns:
        Dictionary of metric distributions with proper error handling
    """
    metrics_lists = {
        'density_uniformity': [],
        'color_consistency': [],
        'spatial_uniformity': [],
        'iou_score': [],
        'shape_similarity': []
    }
    
    # Handle both dictionary and list inputs
    results_items = results.items() if isinstance(results, dict) else enumerate(results)
    
    for _, product_metrics in results_items:
        try:
            if isinstance(product_metrics, dict) and 'reconstruction_metrics' in product_metrics:
                recon = product_metrics['reconstruction_metrics']
                
                # Quality metrics
                if recon and isinstance(recon, dict) and 'quality_metrics' in recon:
                    quality = recon['quality_metrics']
                    if quality and isinstance(quality, dict):
                        for metric in ['density_uniformity', 'color_consistency', 'spatial_uniformity']:
                            if metric in quality and quality[metric] is not None:
                                metrics_lists[metric].append(float(quality[metric]))
                
                # Depth metrics
                if recon and isinstance(recon, dict) and 'depth_metrics' in recon:
                    depth = recon['depth_metrics']
                    if depth and isinstance(depth, dict):
                        if 'silhouette_metrics' in depth:
                            sil = depth['silhouette_metrics']
                            if 'iou_score' in sil:
                                metrics_lists['iou_score'].append(float(sil['iou_score']))
                                
                        if 'contour_metrics' in depth:
                            cont = depth['contour_metrics']
                            if 'shape_similarity' in cont:
                                metrics_lists['shape_similarity'].append(float(cont['shape_similarity']))
                                
        except Exception as e:
            print(f"Warning: Error processing metrics for distribution: {str(e)}")
            continue
    
    # Calculate distributions with safety checks
    distributions = {}
    for metric, values in metrics_lists.items():
        if values:
            distributions[metric] = {
                'min': float(np.min(values)),
                'max': float(np.max(values)),
                'mean': float(np.mean(values)),
                'std': float(np.std(values))
            }
        else:
            distributions[metric] = {
                'min': 0.0,
                'max': 0.0,
                'mean': 0.0,
                'std': 0.0
            }
    
    return distributions

def load_product_metadata(csv_path):
    """
    Loads product metadata from CSV file containing product IDs and categories.
    
    Args:
        csv_path: Path to the CSV file
    Returns:
        Dictionary mapping product IDs to their metadata
    """
    import pandas as pd
    
    product_metadata = {}
    try:
        df = pd.read_csv(csv_path, header=None, 
                        names=['product_id', 'category', 'front_path', 'back_path', 'top_path'])
        
        for _, row in df.iterrows():
            product_metadata[row['product_id']] = {
                'category': row['category'],
                'paths': {
                    'front': row['front_path'],
                    'back': row['back_path'],
                    'top': row['top_path'] if pd.notna(row['top_path']) else None
                }
            }
        return product_metadata
    except Exception as e:
        print(f"Error loading product metadata: {str(e)}")
        return None
    
@timer_decorator
def process_dataset(csv_path, output_path):
    """
    Process all products with comprehensive error handling and reporting.
    """
    try:
        output_path = Path(output_path)
        output_path.mkdir(parents=True, exist_ok=True)
        
        # Load product metadata
        product_metadata = load_product_metadata(csv_path)
        if not product_metadata:
            raise ValueError("Failed to load product metadata")
        
        all_results = {}
        failed_products = []
        
        print(f"\nProcessing {len(product_metadata)} products...")
        
        for i, (product_id, metadata) in enumerate(product_metadata.items(), 1):
            print(f"\nProcessing product {i}/{len(product_metadata)}: {product_id}")
            
            try:
                product_dir = Path(metadata['paths']['front']).parent
                
                results = process_single_product(
                    metadata['paths']['front'],
                    metadata['paths']['back'],
                    metadata['paths'].get('top'),
                    product_dir
                )
                
                if results:
                    results['category'] = metadata['category']
                    all_results[product_id] = results
                else:
                    failed_products.append(product_id)
                    
            except Exception as e:
                print(f"Error processing product {product_id}: {str(e)}")
                failed_products.append(product_id)
                continue
        
        # Generate summary
        if all_results:
            generate_dataset_summary(all_results, output_path, product_metadata)
            
        # Generate error report
        if failed_products:
            with open(output_path / 'failed_products.txt', 'w') as f:
                f.write(f"Failed to process {len(failed_products)} products:\n")
                for prod_id in failed_products:
                    f.write(f"- {prod_id}\n")
        
        return all_results
        
    except Exception as e:
        print(f"Critical error in process_dataset: {str(e)}")
        return None

# Example usage
if __name__ == "__main__":
    csv_path = Path("../product_views.csv")
    output_path = Path("../dataset")
    
    print("Starting dataset processing...")
    results = process_dataset(csv_path, output_path)
    
    print("\nProcessing complete!")
    print(f"Results saved to: {output_path}")


Processing product 3/171: B0009JRWR6

Loading and normalizing images...
Front view background color (RGB): (255, 254, 255)
Front view normalized size: 128x128
Back view background color (RGB): (255, 254, 255)
Back view normalized size: 128x128
Top view background color (RGB): (255, 254, 255)
Top view normalized size: 128x128
Loading image...
It's already a numpy array - verify format
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 254, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...

After background removal:
Result RGB shape: (128, 128, 3)
Alpha mask shape: (128, 128)
Non-zero alpha pixels: 4873

Depth map statistics:
Depth map shape: (128, 128)
Depth range: 36.000 to 254.000

After resizing:
Resized depth map shape: (128, 128)
Valid mask points: 4873

Normalized depth range: 0.000 to 1.000

Point generation results:
Total possible points: 16384
Points after masking: 4

Traceback (most recent call last):
  File "C:\Users\ACER\AppData\Local\Temp\ipykernel_13556\3205619587.py", line 399, in process_single_product
    with open(metrics_file, 'w') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\interactiveshell.py", line 310, in _modified_open
    return io_open(file, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '..\\datasets\\product_95\\evaluation_metrics.json'



Depth map statistics:
Depth map shape: (128, 128)
Depth range: 40.000 to 255.000

After resizing:
Resized depth map shape: (128, 128)
Valid mask points: 1086

Normalized depth range: 0.000 to 1.000

Point generation results:
Total possible points: 16384
Points after masking: 1086
Final points after depth threshold: 1086

=== Mesh Creation Complete ===
Outliers removed: 17
Loading image...
It's already a numpy array - verify format
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (253, 254, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...

After background removal:
Result RGB shape: (128, 128, 3)
Alpha mask shape: (128, 128)
Non-zero alpha pixels: 1086

Depth map statistics:
Depth map shape: (128, 128)
Depth range: 40.000 to 255.000

After resizing:
Resized depth map shape: (128, 128)
Valid mask points: 1086

Normalized depth range: 0.000 to 1.000

Point generation results:
Tota

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (2) found smaller than n_clusters (3). Possibly due to duplicate points in X.

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (2) found smaller than n_clusters (3). Possibly due to duplicate points in X.

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.



Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...

After background removal:
Result RGB shape: (128, 128, 3)
Alpha mask shape: (128, 128)
Non-zero alpha pixels: 4379

Depth map statistics:
Depth map shape: (128, 128)
Depth range: 11.000 to 255.000

After resizing:
Resized depth map shape: (128, 128)
Valid mask points: 4379

Normalized depth range: 0.000 to 1.000

Point generation results:
Total possible points: 16384
Points after masking: 4379
Final points after depth threshold: 4350

=== Mesh Creation Complete ===
Outliers removed: 56
Loading image...
It's already a numpy array - verify format
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...

After background removal:
Result RGB shape: (128, 128, 3)
Alpha m

...


Depth map statistics:
Depth map shape: (128, 128)
Depth range: 49.000 to 255.000

After resizing:
Resized depth map shape: (128, 128)
Valid mask points: 1091

Normalized depth range: 0.000 to 1.000

Point generation results:
Total possible points: 16384
Points after masking: 1091
Final points after depth threshold: 1091

=== Mesh Creation Complete ===
Outliers removed: 16
Loading image...
It's already a numpy array - verify format
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...

After background removal:
Result RGB shape: (128, 128, 3)
Alpha mask shape: (128, 128)
Non-zero alpha pixels: 1091


c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.




Depth map statistics:
Depth map shape: (128, 128)
Depth range: 49.000 to 255.000

After resizing:
Resized depth map shape: (128, 128)
Valid mask points: 1091

Normalized depth range: 0.000 to 1.000

Point generation results:
Total possible points: 16384
Points after masking: 1091
Final points after depth threshold: 1091

=== Mesh Creation Complete ===
Outliers removed: 16

=== Starting Mesh Combination ===

Front view initial points: 1075

Back view initial points: 1075

Sampling Front view:
Original points: 1075
Target points: 1075

Sampling Back view:
Original points: 1075
Target points: 1075

Dimensions:
Original width: 96.0
Original height: 105.0
Aspect ratio: 0.9142857142857143

Normalizing depths:
Front z-range: 0.267 to 0.655
Back z-range: 0.267 to 0.655

Final combined point count: 2150
Combined z-range: 0.250 to 0.750

=== Mesh Combination Complete ===

Fill Results:
Front fill points: 8267
Back fill points: 8635

Processing product 167/171: B014X2BFTA

Loading and normalizin

Traceback (most recent call last):
  File "C:\Users\ACER\AppData\Local\Temp\ipykernel_13556\3205619587.py", line 399, in process_single_product
    with open(metrics_file, 'w') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\interactiveshell.py", line 310, in _modified_open
    return io_open(file, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '..\\datasets\\product_4533\\evaluation_metrics.json'


Detected background color (RGB): (254, 253, 253)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...

After background removal:
Result RGB shape: (128, 128, 3)
Alpha mask shape: (128, 128)
Non-zero alpha pixels: 9194

Depth map statistics:
Depth map shape: (128, 128)
Depth range: 17.000 to 255.000

After resizing:
Resized depth map shape: (128, 128)
Valid mask points: 9194

Normalized depth range: 0.000 to 1.000

Point generation results:
Total possible points: 16384
Points after masking: 9194
Final points after depth threshold: 9194

=== Mesh Creation Complete ===
Outliers removed: 96
Loading image...
It's already a numpy array - verify format
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (254, 253, 253)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...

After background removal:
Result RGB shape: (128, 128, 3)
Alpha m